# Enriquecimiento de la Capa Plata

En esta notebook trabajaremos sobre los datos de telemetría solar generados en el paso anterior. El objetivo es **enriquecer y completar la serie temporal**, detectando brechas (gaps) de tiempo donde los controladores perdieron conexión, y aplicando técnicas de imputación antes de pasarlos a modelos analíticos.

## Importar las librerías necesarias e inicializar paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)
pd.set_option('display.max_columns', None)

BASE_DIR = Path('..').resolve()
PLATA_DIR = BASE_DIR / 'data' / 'plata'

archivo_base = PLATA_DIR / "dataset_telemetria_limpio.csv"
archivo_diario = PLATA_DIR / "dataset_telemetria_diario.csv"

print("Librerías importadas y paths configurados.")

Librerías importadas y paths configurados.


## Carga de Datasets y Verificación

In [ ]:
try:
    df_base = pd.read_csv(archivo_base, parse_dates=["timestamp", "FECHA"])
    print(f"Dataset base cargado: {len(df_base)} registros")
except FileNotFoundError:
    print("No se encontró el dataset base.")
    df_base = pd.DataFrame()

try:
    df_diario = pd.read_csv(archivo_diario, parse_dates=["FECHA"])
    print(f"Dataset diario cargado: {len(df_diario)} registros")
except FileNotFoundError:
    print("No se encontró el dataset diario.")
    df_diario = pd.DataFrame()


## Detección y Análisis de Fechas Faltantes (Dataset Diario)

Los sistemas IoT pueden estar apagados días enteros. Vamos a asegurar que cada estación tenga una serie de tiempo continua, rellenando los días faltantes con filas vacías que imputaremos luego.

In [ ]:
if not df_diario.empty:
    # Rango completo desde el primer hasta el último día registrado en general
    fechas_totales = pd.date_range(start=df_diario['FECHA'].min(), end=df_diario['FECHA'].max(), freq='D')
    estaciones = df_diario['station_name'].unique()
    
    # Crear un índice con todas las combinaciones posibles
    index_completo = pd.MultiIndex.from_product([estaciones, fechas_totales], names=['station_name', 'FECHA'])
    
    # Reindexar el DataFrame
    df_diario_continuo = df_diario.set_index(['station_name', 'FECHA']).reindex(index_completo).reset_index()
    
    dias_agregados = len(df_diario_continuo) - len(df_diario)
    print(f"Se detectaron y agregaron {dias_agregados} días vacíos (faltantes) en la serie temporal para asegurar continuidad.")


## Tratamiento de Valores Nulos (Dataset Diario)

Para datos de batería (SOC) o energía, si faltan días completos, haremos una interpolación lineal direccional en el tiempo, para rellenar los vacíos generados en el paso anterior.

In [ ]:
if not df_diario.empty:
    print("Nulos por columna antes de la imputación:")
    print(df_diario_continuo.isnull().sum())
    
    # Ordenar estrictamente por tiempo dentro de cada estación
    df_diario_continuo = df_diario_continuo.sort_values(by=['station_name', 'FECHA'])
    
    # Seleccionamos solo las columnas numéricas para interpolar
    cols_numericas = df_diario_continuo.select_dtypes(include=[np.number]).columns
    
    # Aplicamos interpolación lineal agrupada por estación
    for col in cols_numericas:
        df_diario_continuo[col] = df_diario_continuo.groupby('station_name')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
        
    # Si una estación completa era nula desde el inicio y quedó con NaNs, la rellenamos con la media global
    df_diario_continuo[cols_numericas] = df_diario_continuo[cols_numericas].fillna(df_diario_continuo[cols_numericas].mean())
    
    print("\nNulos después de la imputación:")
    print(df_diario_continuo.isnull().sum())


## Detección de Brechas y Remuestreo (Dataset Base de Alta Frecuencia)

La telemetría original no siempre se envía en la misma frecuencia exacta de segundos. Vamos a regularizar la serie a **intervalos fijos de 10 minutos** (`10min`). Esto alinea los timestamps de todas las estaciones.

In [ ]:
if not df_base.empty:
    # Aseguramos que el timestamp sea el índice
    df_base.set_index('timestamp', inplace=True)
    
    # Agrupamos por estación y remuestreamos ('10min') usando el promedio en ese segmento
    df_base_resampled = df_base.groupby('station_name').resample('10min').mean(numeric_only=True).reset_index()
    
    print(f"Filas en dataset original: {len(df_base)}")
    print(f"Filas tras crear frecuencia regular de 10 minutos: {len(df_base_resampled)}")


## Imputación Inteligente de Telemetría Solar

Al regularizar a 10 minutos, se generaron valores nulos (`NaN`) en los segmentos donde el sensor no mandó señal. Aplicaremos reglas específicas para sistemas solares:
1. **Variables PV (Paneles):** Si es de noche (entre las 19:00 y las 05:00), es físicamente imposible que haya voltaje o potencia. Se rellenan con 0. Durante el día, los gaps se interpolan.
2. **Variables de Batería/Carga:** Se interpolan según el tiempo, ya que continúan operando de día y de noche.

In [ ]:
if not df_base.empty:
    # Extraer la hora para aplicar reglas de día/noche
    df_base_resampled['hora'] = df_base_resampled['timestamp'].dt.hour
    
    # Interpolación de tiempo general para batería, temperatura, consumos
    cols_to_interpolate = [c for c in df_base_resampled.columns if 'battery' in c or 'temp' in c or 'load' in c]
    for col in cols_to_interpolate:
        # Se usa interpolación lineal basada en tiempo
        df_base_resampled[col] = df_base_resampled.groupby('station_name')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
    
    # Imputación guiada por reglas para paneles (PV)
    cols_pv = [c for c in df_base_resampled.columns if 'pv_' in c]
    for col in cols_pv:
        # 1. Rellenar con 0 durante el bloque nocturno si el sensor estaba desconectado
        is_night = (df_base_resampled['hora'] >= 19) | (df_base_resampled['hora'] <= 5)
        df_base_resampled.loc[is_night & df_base_resampled[col].isna(), col] = 0
        
        # 2. Interpolar los saltos durante las horas de sol
        df_base_resampled[col] = df_base_resampled.groupby('station_name')[col].transform(lambda x: x.interpolate(method='linear', limit_direction='both'))
        
    # Limpieza de columnas temporales
    df_base_resampled.drop(columns=['hora'], inplace=True)
    
    print("Imputación guiada por dominio de negocio (solar) completada.")


## Exportar Capa Plata Enriquecida

In [ ]:
if not df_diario.empty and not df_base.empty:
    path_base_enriquecido = PLATA_DIR / "dataset_telemetria_base_enriquecido.csv"
    path_diario_enriquecido = PLATA_DIR / "dataset_telemetria_diario_enriquecido.csv"
    
    df_base_resampled.to_csv(path_base_enriquecido, index=False)
    df_diario_continuo.to_csv(path_diario_enriquecido, index=False)
    
    print("\nDatasets enriquecidos exportados correctamente a la Capa Plata:")
    print(f"- {path_base_enriquecido}")
    print(f"- {path_diario_enriquecido}")


# Conclusión

En esta etapa hemos completado el **enriquecimiento de datos**, crucial para los modelos analíticos o de Machine Learning:

1. **Series Temporales Continuas**: Reindexamos los datos para cubrir todos los días sin huecos silenciosos.
2. **Resampling Estandarizado**: Normalizamos el dataset base de eventos irregulares a intervalos rígidos y predictibles (cada 10 minutos).
3. **Imputación con Lógica Física**: Aplicamos reglas del dominio de la energía renovable (forzando valores cero en las noches sin registro) para no distorsionar las curvas de potencia de los controladores.